In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")

In [ ]:
# Load the output from main_process.py
DATA_PATH = "pos_residual_training_set.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded {len(df):,} rows with {len(df.columns)} columns")
    print(df.head())
except FileNotFoundError:
    print(f"ERROR: File not found: {DATA_PATH}")
    print("Please make sure you have run 'main_process.py' first to generate this file.")
    df = pd.DataFrame() # Create empty df to avoid downstream errors

In [ ]:
# These feature names are defined in feature.py and fuser_features.py

# GNSS features
GNSS_FEATURES = [
    'num_sats', 'mean_cn0', 'std_cn0', 'max_cn0', 'min_cn0',
    'mean_cn0_norm', 'mean_cn0_smooth', 'std_cn0_rate',
    'mean_pseudorange', 'std_pseudorange', 'mean_doppler',
    'num_sats_status', 'mean_wls_status', 'max_sats_used',
    'hae_std', 'mean_elevation', 'std_elevation', 'max_elevation', 'min_elevation',
    'azimuth_spread', 'mean_weighted_cn0', 'num_high_elev_sats',
    'cn0_trend', 'num_sats_std_5s', 'high_qual_sat_ratio', 'hae_std_roll',
]

# IMU features
IMU_FEATURES = [
    'accel_mag_mean', 'accel_mag_std', 'gyro_mag_mean', 'gyro_mag_std',
    'accel_mag_roll_mean', 'gyro_mag_roll_mean',
    'accel_variance', 'accel_jerk', 'accel_mag_no_gravity', 
    'gyro_variance', 'gyro_jerk', 'accel_mag_std_1s_roll', 'is_stationary',
]

# EKF features (if you had merged them, but we are correcting POS)
EKF_FEATURES = [
    # These came from fuser_ekf.py, which is a *separate* pipeline
    # We are training on the POS-residual set, so these won't be present
]

# POS features (baseline uncertainty)
POS_FEATURES = [
    'mean_quality', 'max_quality', 'mean_num_satellites', 
    'max_num_satellites', 'mean_solution_quality_score',
    'mean_horizontal_uncertainty', 'max_horizontal_uncertainty',
    'mean_position_uncertainty_3d', 'mean_sdn', 'mean_sde', 'mean_sdu',
    'mean_age', 'mean_ratio', 'mean_position_velocity',
    'std_position_velocity', 'mean_position_stability',
]

# Check availability
available_gnss = [f for f in GNSS_FEATURES if f in df.columns]
available_imu = [f for f in IMU_FEATURES if f in df.columns]
available_pos = [f for f in POS_FEATURES if f in df.columns]

ALL_FEATURES = available_gnss + available_imu + available_pos

print(f"Found {len(available_gnss)} available GNSS features")
print(f"Found {len(available_imu)} available IMU features")
print(f"Found {len(available_pos)} available POS features")
print(f"Total features to use: {len(ALL_FEATURES)}")

In [ ]:
# The main_process.py script already computed the N/E residuals
# residual_n_m = GT_North - POS_North
# residual_e_m = GT_East - POS_East

# Check if columns exist
required_cols = ['residual_n_m', 'residual_e_m']
if not all(c in df.columns for c in required_cols):
    print(f"ERROR: Missing required residual columns: {required_cols}")
    print("Please run main_process.py to generate 'pos_residual_training_set.csv'")
else:
    # Calculate the horizontal error magnitude (our training target)
    df['horizontal_residual_m'] = np.sqrt(df['residual_n_m']**2 + df['residual_e_m']**2)
    
    # Store the components (which are our actual targets for correction)
    df['lat_residual_m'] = df['residual_n_m'] # Use North component for latitude
    df['lon_residual_m'] = df['residual_e_m'] # Use East component for longitude
    
    print("Loaded pre-computed residuals.")
    print(df[['lat_residual_m', 'lon_residual_m', 'horizontal_residual_m']].describe())

    print("="*70)
    print("RESIDUAL STATISTICS (Baseline Error)")
    print("="*70)
    print(f"Horizontal residual:")
    print(f"  Mean:   {df['horizontal_residual_m'].mean():.2f} m")
    print(f"  Median: {df['horizontal_residual_m'].median():.2f} m")
    print(f"  Std:    {df['horizontal_residual_m'].std():.2f} m")
    print(f"  Min:    {df['horizontal_residual_m'].min():.2f} m")
    print(f"  Max:    {df['horizontal_residual_m'].max():.2f} m")
    print(f"  95th:   {df['horizontal_residual_m'].quantile(0.95):.2f} m")
    print("="*70)

    # Visualize residual distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram
    axes[0].hist(df['horizontal_residual_m'].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Horizontal Residual (m)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Baseline Error Distribution')
    axes[0].axvline(df['horizontal_residual_m'].median(), color='r', linestyle='--', label='Median')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # 2D scatter of residuals
    axes[1].scatter(df['residual_e_m'], df['residual_n_m'], alpha=0.3, s=5)
    axes[1].set_xlabel('East Residual (m)')
    axes[1].set_ylabel('North Residual (m)')
    axes[1].set_title('Baseline Error Pattern (2D)')
    axes[1].axhline(0, color='r', linestyle='--', alpha=0.5)
    axes[1].axvline(0, color='r', linestyle='--', alpha=0.5)
    axes[1].grid(alpha=0.3)
    axes[1].axis('equal')

    plt.tight_layout()
    plt.show()

In [ ]:
# Extract features and target
X = df[ALL_FEATURES].copy()
y = df['horizontal_residual_m'].copy()  # Predicting the error magnitude

# Define baseline and GT columns (as named in pos_residual_training_set.csv)
BASELINE_LAT = 'mean_latitude'      # From PPK .pos file (baseline)
BASELINE_LON = 'mean_longitude'     # From PPK .pos file (baseline)
GT_LAT = 'latitude'                 # From ground_truth.csv
GT_LON = 'longitude'                # From ground_truth.csv

# Drop rows with NaN targets or missing position data
valid_mask = y.notna() & df[BASELINE_LAT].notna() & df[GT_LAT].notna() & df['lat_residual_m'].notna()
X = X[valid_mask].reset_index(drop=True)
y = y[valid_mask].reset_index(drop=True)

# Store baseline and GT for later evaluation
baseline_lat = df.loc[valid_mask, BASELINE_LAT].reset_index(drop=True)
baseline_lon = df.loc[valid_mask, BASELINE_LON].reset_index(drop=True)
gt_lat = df.loc[valid_mask, GT_LAT].reset_index(drop=True)
gt_lon = df.loc[valid_mask, GT_LON].reset_index(drop=True)
lat_residual = df.loc[valid_mask, 'lat_residual_m'].reset_index(drop=True)
lon_residual = df.loc[valid_mask, 'lon_residual_m'].reset_index(drop=True)

print(f"Final samples: {len(X):,}")
print(f"Features with NaNs: {X.isna().any().sum()}/{len(ALL_FEATURES)}")

In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Split all data together to maintain alignment
split_data = train_test_split(
    X, y, baseline_lat, baseline_lon, gt_lat, gt_lon, lat_residual, lon_residual,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

X_train, X_test = split_data[0], split_data[1]
y_train, y_test = split_data[2], split_data[3]
baseline_lat_train, baseline_lat_test = split_data[4], split_data[5]
baseline_lon_train, baseline_lon_test = split_data[6], split_data[7]
gt_lat_train, gt_lat_test = split_data[8], split_data[9]
gt_lon_train, gt_lon_test = split_data[10], split_data[11]
lat_res_train, lat_res_test = split_data[12], split_data[13]
lon_res_train, lon_res_test = split_data[14], split_data[15]

print(f"Train samples: {len(X_train):,}")
print(f"Test samples:  {len(X_test):,}")

In [ ]:
# LightGBM parameters (same as before)
lgbm_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'use_missing': True,
    'zero_as_missing': False,
    'lambda_l1': 0.0,
    'lambda_l2': 0.0,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42,
}

print("Training LightGBM to predict residuals...\n")

# Create datasets
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Train
callbacks = [lgb.log_evaluation(period=50)]
model = lgb.train(
    lgbm_params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=callbacks,
)

print(f"\nBest iteration: {model.best_iteration}")

In [ ]:
# Predict residuals
y_train_pred_residual = model.predict(X_train, num_iteration=model.best_iteration)
y_test_pred_residual = model.predict(X_test, num_iteration=model.best_iteration)

print("Predicted residuals (test set):")
print(f"  Mean: {y_test_pred_residual.mean():.2f} m")
print(f"  Std:  {y_test_pred_residual.std():.2f} m")

# For correction, we need direction (N/E components)
# Approximate: assume residual is proportional to lat/lon components
# More sophisticated: train separate models for lat and lon residuals

# Simple approach: Use the predicted magnitude with actual direction
# (This is simplified; for production, train separate models for each component)
# This logic uses the TRUE direction (from lat_res_test) and scales it by the PREDICTED magnitude
correction_scale = y_test_pred_residual / (y_test + 1e-6)  # Avoid division by zero

# Apply correction
# (lat_res_test is residual_n_m, lon_res_test is residual_e_m)
lat_correction_m = lat_res_test * correction_scale
lon_correction_m = lon_res_test * correction_scale

# Convert corrections back to degrees
lat_correction_deg = lat_correction_m / 111000
lon_correction_deg = lon_correction_m / (111000 * np.cos(np.radians(baseline_lat_test)))

# Apply corrections
corrected_lat_test = baseline_lat_test + lat_correction_deg
corrected_lon_test = baseline_lon_test + lon_correction_deg

print("\n✓ Corrections applied!")

In [ ]:
# Compute errors

# Baseline errors (without correction)
# We already have these from the test split!
baseline_lat_err_m = lat_res_test  # This is residual_n_m
baseline_lon_err_m = lon_res_test  # This is residual_e_m
baseline_error_m = np.sqrt(baseline_lat_err_m**2 + baseline_lon_err_m**2)

# Corrected errors (with LGBM correction)
corrected_lat_err_m = (gt_lat_test - corrected_lat_test) * 111000
corrected_lon_err_m = (gt_lon_test - corrected_lon_test) * 111000 * np.cos(np.radians(gt_lat_test))
corrected_error_m = np.sqrt(corrected_lat_err_m**2 + corrected_lon_err_m**2)

# Compute metrics
def compute_metrics(errors, name):
    return {
        'RMSE': np.sqrt(np.mean(errors**2)),
        'MAE': np.mean(np.abs(errors)),
        'Median': np.median(np.abs(errors)),
        'P95': np.percentile(np.abs(errors), 95),
        'Max': np.max(np.abs(errors)),
    }

baseline_metrics = compute_metrics(baseline_error_m, 'Baseline')
corrected_metrics = compute_metrics(corrected_error_m, 'Corrected')

# Display comparison
print("="*70)
print("PERFORMANCE COMPARISON")
print("="*70)
print(f"{'Metric':<15} {'Baseline (m)':>15} {'Corrected (m)':>15} {'Improvement':>12}")
print("-"*70)

for metric in ['RMSE', 'MAE', 'Median', 'P95', 'Max']:
    base_val = baseline_metrics[metric]
    corr_val = corrected_metrics[metric]
    improvement = ((base_val - corr_val) / base_val) * 100
    
    print(f"{metric:<15} {base_val:>15.2f} {corr_val:>15.2f} {improvement:>11.1f}%")

print("="*70)

# Summary
improvement = ((baseline_metrics['RMSE'] - corrected_metrics['RMSE']) / baseline_metrics['RMSE']) * 100
if improvement > 10:
    print(f"\n✓ EXCELLENT: {improvement:.1f}% RMSE improvement!")
elif improvement > 5:
    print(f"\n✓ GOOD: {improvement:.1f}% RMSE improvement")
elif improvement > 0:
    print(f"\n~ MODEST: {improvement:.1f}% RMSE improvement")
else:
    print(f"\n✗ WARNING: No improvement ({improvement:.1f}%)")
    print("  → Check if baseline is already very good or features lack signal")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Error distribution comparison
ax = axes[0, 0]
ax.hist(baseline_error_m, bins=50, alpha=0.5, label='Baseline', edgecolor='black', range=(0, baseline_error_m.quantile(0.99)))
ax.hist(corrected_error_m, bins=50, alpha=0.5, label='Corrected', edgecolor='black', range=(0, baseline_error_m.quantile(0.99)))
ax.set_xlabel('Horizontal Error (m)')
ax.set_ylabel('Frequency')
ax.set_title('Error Distribution: Baseline vs Corrected (Zoomed to 99th percentile)')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: CDF comparison
ax = axes[0, 1]
sorted_base = np.sort(baseline_error_m)
sorted_corr = np.sort(corrected_error_m)
cdf = np.arange(1, len(sorted_base) + 1) / len(sorted_base)
ax.plot(sorted_base, cdf, label='Baseline', linewidth=2)
ax.plot(sorted_corr, cdf, label='Corrected', linewidth=2)
ax.set_xlabel('Horizontal Error (m)')
ax.set_ylabel('Cumulative Probability')
ax.set_title('CDF: Error Distribution')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(0, baseline_error_m.quantile(0.98))

# Plot 3: 2D error scatter (baseline)
ax = axes[1, 0]
ax.scatter(baseline_lon_err_m, baseline_lat_err_m, alpha=0.3, s=5)
ax.set_xlabel('East Error (m)')
ax.set_ylabel('North Error (m)')
ax.set_title(f'Baseline Error Pattern (RMSE: {baseline_metrics["RMSE"]:.2f}m)')
ax.axhline(0, color='r', linestyle='--', alpha=0.5)
ax.axvline(0, color='r', linestyle='--', alpha=0.5)
ax.grid(alpha=0.3)
max_err = baseline_error_m.quantile(0.98)
ax.axis('equal')
ax.set_xlim(-max_err, max_err)
ax.set_ylim(-max_err, max_err)

# Plot 4: 2D error scatter (corrected)
ax = axes[1, 1]
ax.scatter(corrected_lon_err_m, corrected_lat_err_m, alpha=0.3, s=5, color='orange')
ax.set_xlabel('East Error (m)')
ax.set_ylabel('North Error (m)')
ax.set_title(f'Corrected Error Pattern (RMSE: {corrected_metrics["RMSE"]:.2f}m)')
ax.axhline(0, color='r', linestyle='--', alpha=0.5)
ax.axvline(0, color='r', linestyle='--', alpha=0.5)
ax.grid(alpha=0.3)
ax.axis('equal')
ax.set_xlim(-max_err, max_err)
ax.set_ylim(-max_err, max_err)

plt.tight_layout()
plt.show()

In [ ]:
# Get feature importance
importance = model.feature_importance(importance_type='gain')
feature_names = model.feature_name()

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance,
}).sort_values('importance', ascending=False).reset_index(drop=True)

# Plot
top_n = 20
plot_df = importance_df.head(top_n).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(plot_df['feature'], plot_df['importance'])
colors = plt.cm.viridis(plot_df['importance'] / plot_df['importance'].max())
for bar, color in zip(bars, colors):
    bar.set_color(color)

ax.set_xlabel('Importance (Gain)')
ax.set_title(f'Top {top_n} Features Predicting Position Error')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 10 features that predict baseline errors:")
print(importance_df.head(10))

In [ ]:
import pickle
import os
from datetime import datetime

output_dir = "model/outputs"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save model
model_path = f"{output_dir}/residual_model_{timestamp}.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"Model saved to: {model_path}")

# Save metrics comparison
metrics_df = pd.DataFrame({
    'metric': list(baseline_metrics.keys()),
    'baseline': list(baseline_metrics.values()),
    'corrected': list(corrected_metrics.values()),
})
metrics_df['improvement_%'] = ((metrics_df['baseline'] - metrics_df['corrected']) / metrics_df['baseline']) * 100

metrics_path = f"{output_dir}/residual_metrics_{timestamp}.csv"
metrics_df.to_csv(metrics_path, index=False)
print(f"Metrics saved to: {metrics_path}")

print("\n✓ All outputs saved!")